In [1]:
#cell 1
# Install runtime dependencies for Colab + A100 + vLLM.

!pip -q install -U uv

!uv pip install --system -U openai requests tqdm jsonschema psutil numpy pandas accelerate safetensors

# Remove optional packages that may break transformers/vLLM imports in Colab.
!uv pip uninstall --system -y torchcodec torchvision torchaudio sentence-transformers || true

# Transformers is used only for tokenizer-based input truncation.
!uv pip install --system -U "transformers>=4.51.0"

# Qwen3.5 requires a recent vLLM build.
# For A100, do not force CUDA 13 / cu130. Let uv choose the compatible torch backend.
!uv pip install --system -U vllm --torch-backend=auto --extra-index-url https://wheels.vllm.ai/nightly

import sys
import importlib.metadata as md

import torch
import vllm
import transformers

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("vLLM:", vllm.__version__)
print("transformers:", transformers.__version__)

for pkg in ["torchcodec", "torchvision", "torchaudio", "sentence-transformers"]:
    try:
        print(pkg + ":", md.version(pkg))
    except Exception:
        print(pkg + ": not installed")

!nvidia-smi

Using Python 3.12.13 environment at: /usr
Resolved 71 packages in 163ms
Prepared 8 packages in 0.67ms
Uninstalled 8 packages in 255ms
Installed 8 packages in 216ms
 - numpy==2.3.5
 + numpy==2.4.6
 - nvidia-cublas==13.1.0.3
 + nvidia-cublas==13.1.1.3
 - nvidia-cudnn-cu13==9.19.0.56
 + nvidia-cudnn-cu13==9.20.0.48
 - nvidia-cusparselt-cu13==0.8.0
 + nvidia-cusparselt-cu13==0.8.1
 - nvidia-nccl-cu13==2.28.9
 + nvidia-nccl-cu13==2.29.7
 - setuptools==80.10.2
 + setuptools==81.0.0
 - torch==2.11.0+cu130
 + torch==2.12.0
 - triton==3.6.0
 + triton==3.7.0
Using Python 3.12.13 environment at: /usr
Uninstalled 2 packages in 104ms
 - torchaudio==2.11.0+cu130
 - torchvision==0.26.0+cu130
Using Python 3.12.13 environment at: /usr
Resolved 27 packages in 107ms
Checked 27 packages in 0.56ms
Using Python 3.12.13 environment at: /usr
Resolved 189 packages in 21.97s
Prepared 10 packages in 20ms
Uninstalled 8 packages in 216ms
Installed 10 packages in 216ms
 - numpy==2.4.6
 + numpy==2.3.5
 - nvidia-cubl

In [2]:
#cell 2
# Imports and global configuration for BM25 evidence answering on A100.

import os
import re
import gc
import json
import time
import shlex
import shutil
import psutil
import subprocess
import traceback

from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI
from transformers import AutoTokenizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"

LLM_MODEL_NAME = "Qwen/Qwen3.5-27B"

PORT = 8000
BASE_URL = f"http://localhost:{PORT}/v1"

# The user prompt is capped around 8192 tokens.
MAX_INPUT_TOKENS = 8192

# This is the generation budget.
ANSWER_MAX_TOKENS = 256

# Total model context length = input + output + margin.
# This is conservative for A100 80GB and your BM25 QA setup.
MAX_MODEL_LEN = 10240

# A100 80GB conservative single-request settings.
GPU_MEMORY_UTILIZATION = 0.90
MAX_NUM_SEQS = 1
MAX_NUM_BATCHED_TOKENS = MAX_MODEL_LEN

SERVER_LOG_PATH = Path("/content/vllm_bm25_answer_server.log")
SERVER_PID_PATH = Path("/content/vllm_bm25_answer_server.pid")

PROJECT_DIR = Path("/content/drive/MyDrive/final_project")
BM25_DIR = PROJECT_DIR / "BM25"
DRIVE_EVIDENCE_DIR = BM25_DIR / "evidence"
DRIVE_ANSWER_DIR = BM25_DIR / "answer"

LOCAL_RUNTIME_DIR = Path("/content/final_project_bm25_answer_copy")
LOCAL_EVIDENCE_DIR = LOCAL_RUNTIME_DIR / "evidence"
LOCAL_EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = {
    "hotpotqa": {
        "drive_evidence_path": DRIVE_EVIDENCE_DIR / "hotpotqa_evidence.json",
        "local_evidence_path": LOCAL_EVIDENCE_DIR / "hotpotqa_evidence.json",
        "answer_path": DRIVE_ANSWER_DIR / "hotpotqa_qwen3.5_bm25_answers.json",
    },
    "2wikimultihopqa": {
        "drive_evidence_path": DRIVE_EVIDENCE_DIR / "2wikimultihopqa_evidence.json",
        "local_evidence_path": LOCAL_EVIDENCE_DIR / "2wikimultihopqa_evidence.json",
        "answer_path": DRIVE_ANSWER_DIR / "2wikimultihopqa_qwen3.5_bm25_answers.json",
    },
}

EXPECTED_NUM_RECORDS_PER_DATASET = 1000

ANSWER_START_INDEX = 0
ANSWER_END_INDEX = None

SAVE_EVERY_N = 1
CLEAR_CACHE_EVERY_N = 25
RESUME_IF_EXISTS = True

print("Model:", LLM_MODEL_NAME)
print("Max input tokens:", MAX_INPUT_TOKENS)
print("Answer max tokens:", ANSWER_MAX_TOKENS)
print("Max model len:", MAX_MODEL_LEN)
print("GPU memory utilization:", GPU_MEMORY_UTILIZATION)
print("BM25 directory:", BM25_DIR)
print("Evidence directory:", DRIVE_EVIDENCE_DIR)
print("Answer directory:", DRIVE_ANSWER_DIR)

for dataset_name, cfg in DATASETS.items():
    print("=" * 80)
    print("Dataset:", dataset_name)
    print("Drive evidence:", cfg["drive_evidence_path"])
    print("Local evidence:", cfg["local_evidence_path"])
    print("Answer output:", cfg["answer_path"])

Model: Qwen/Qwen3.5-27B
Max input tokens: 8192
Answer max tokens: 256
Max model len: 10240
GPU memory utilization: 0.9
BM25 directory: /content/drive/MyDrive/final_project/BM25
Evidence directory: /content/drive/MyDrive/final_project/BM25/evidence
Answer directory: /content/drive/MyDrive/final_project/BM25/answer
Dataset: hotpotqa
Drive evidence: /content/drive/MyDrive/final_project/BM25/evidence/hotpotqa_evidence.json
Local evidence: /content/final_project_bm25_answer_copy/evidence/hotpotqa_evidence.json
Answer output: /content/drive/MyDrive/final_project/BM25/answer/hotpotqa_qwen3.5_bm25_answers.json
Dataset: 2wikimultihopqa
Drive evidence: /content/drive/MyDrive/final_project/BM25/evidence/2wikimultihopqa_evidence.json
Local evidence: /content/final_project_bm25_answer_copy/evidence/2wikimultihopqa_evidence.json
Answer output: /content/drive/MyDrive/final_project/BM25/answer/2wikimultihopqa_qwen3.5_bm25_answers.json


In [3]:
#cell 3
# Mount Google Drive and copy BM25 evidence files to local Colab disk.

from google.colab import drive

MOUNTPOINT = Path("/content/drive")
drive.mount(str(MOUNTPOINT), force_remount=True)

assert PROJECT_DIR.exists(), f"PROJECT_DIR does not exist: {PROJECT_DIR}"
assert BM25_DIR.exists(), f"BM25_DIR does not exist: {BM25_DIR}"
assert DRIVE_EVIDENCE_DIR.exists(), f"Evidence directory does not exist: {DRIVE_EVIDENCE_DIR}"

DRIVE_ANSWER_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

def file_is_same_size(src: Path, dst: Path) -> bool:
    """Check whether the destination file exists and has the same size as the source."""
    return dst.exists() and dst.stat().st_size == src.stat().st_size

def copy_file_to_local(src: Path, dst: Path) -> None:
    """Copy a file to local disk using a temporary file to avoid partial copies."""
    dst.parent.mkdir(parents=True, exist_ok=True)

    if file_is_same_size(src, dst):
        print(f"Local copy already exists: {dst}")
        return

    tmp = dst.with_name(dst.name + ".tmp")
    if tmp.exists():
        tmp.unlink()

    shutil.copy2(src, tmp)
    os.replace(tmp, dst)
    print(f"Copied to local disk: {src} -> {dst}")

for dataset_name, cfg in DATASETS.items():
    drive_path = cfg["drive_evidence_path"]
    local_path = cfg["local_evidence_path"]

    if not drive_path.exists():
        raise FileNotFoundError(f"{dataset_name}: evidence file not found: {drive_path}")

    copy_file_to_local(drive_path, local_path)

    print(f"{dataset_name}: local evidence size MB:", local_path.stat().st_size / (1024 ** 2))

print("BM25 answer directory is ready:", DRIVE_ANSWER_DIR)

Mounted at /content/drive
Copied to local disk: /content/drive/MyDrive/final_project/BM25/evidence/hotpotqa_evidence.json -> /content/final_project_bm25_answer_copy/evidence/hotpotqa_evidence.json
hotpotqa: local evidence size MB: 7.00105094909668
Copied to local disk: /content/drive/MyDrive/final_project/BM25/evidence/2wikimultihopqa_evidence.json -> /content/final_project_bm25_answer_copy/evidence/2wikimultihopqa_evidence.json
2wikimultihopqa: local evidence size MB: 5.920868873596191
BM25 answer directory is ready: /content/drive/MyDrive/final_project/BM25/answer


In [4]:
#cell 4
# Load and validate evidence files.

def load_json_list(json_path: Path, dataset_name: str) -> List[Dict[str, Any]]:
    """Load a JSON file whose root must be a list."""
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(f"{dataset_name}: JSON root must be a list.")

    return data

def validate_evidence_record(record: Dict[str, Any], dataset_name: str, index: int) -> None:
    """Validate one evidence record."""
    required_keys = {"type", "question", "answer", "evidence_chunk"}

    if not isinstance(record, dict):
        raise ValueError(f"{dataset_name}: record {index} is not a dictionary.")

    missing = required_keys - set(record.keys())
    if missing:
        raise ValueError(f"{dataset_name}: record {index} is missing keys: {missing}")

    if not isinstance(record["question"], str) or not record["question"].strip():
        raise ValueError(f"{dataset_name}: record {index} has an empty question.")

    if not isinstance(record["evidence_chunk"], list):
        raise ValueError(f"{dataset_name}: record {index} evidence_chunk must be a list.")

    for j, chunk in enumerate(record["evidence_chunk"]):
        if not isinstance(chunk, dict):
            raise ValueError(f"{dataset_name}: record {index}, chunk {j} is not a dictionary.")

        if "title" not in chunk or "text" not in chunk:
            raise ValueError(
                f"{dataset_name}: record {index}, chunk {j} must contain title and text."
            )

def load_and_validate_dataset(dataset_name: str, evidence_path: Path) -> List[Dict[str, Any]]:
    """Load and validate one dataset evidence file."""
    records = load_json_list(evidence_path, dataset_name)

    if len(records) != EXPECTED_NUM_RECORDS_PER_DATASET:
        print(
            f"Warning: {dataset_name} has {len(records)} records, "
            f"expected {EXPECTED_NUM_RECORDS_PER_DATASET}."
        )

    for i, rec in enumerate(records):
        validate_evidence_record(rec, dataset_name, i)

    print("=" * 80)
    print(f"{dataset_name}: validation passed.")
    print("Number of records:", len(records))
    print("First question:", records[0]["question"])
    print("First GT answer:", records[0]["answer"])
    print("First number of chunks:", len(records[0]["evidence_chunk"]))
    print("First chunk keys:", list(records[0]["evidence_chunk"][0].keys()))

    return records

evidence_data = {}

for dataset_name, cfg in DATASETS.items():
    evidence_data[dataset_name] = load_and_validate_dataset(
        dataset_name=dataset_name,
        evidence_path=cfg["local_evidence_path"],
    )

hotpotqa: validation passed.
Number of records: 1000
First question: Which musical fantasy film is older, Bedknobs and Broomsticks or The Muppet Christmas Carol?
First GT answer: Bedknobs and Broomsticks
First number of chunks: 4
First chunk keys: ['title', 'text']
2wikimultihopqa: validation passed.
Number of records: 1000
First question: Which film has the director died earlier, John Jaffer Janardhanan or Kamakalawa?
First GT answer: Kamakalawa
First number of chunks: 4
First chunk keys: ['title', 'text']


In [5]:
#cell 5
# Start Qwen3.5 vLLM server in non-thinking mode on A100.

def kill_process_tree(pid: int) -> None:
    """Kill a process and all child processes."""
    try:
        parent = psutil.Process(int(pid))
        for child in parent.children(recursive=True):
            try:
                child.kill()
            except Exception:
                pass
        parent.kill()
        parent.wait(timeout=10)
        print("Killed process tree:", pid)
    except Exception:
        pass

# Stop old PID from this notebook.
if SERVER_PID_PATH.exists():
    old_pid = SERVER_PID_PATH.read_text().strip()
    if old_pid:
        kill_process_tree(int(old_pid))

# Stop leftover vLLM serve processes.
for p in psutil.process_iter(["pid", "name", "cmdline"]):
    try:
        cmdline = " ".join(p.info.get("cmdline") or [])
        if "vllm" in cmdline and "serve" in cmdline:
            kill_process_tree(p.info["pid"])
            print("Killed leftover vLLM process:", p.info["pid"])
    except Exception:
        pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

time.sleep(3)

cmd = [
    "vllm", "serve", LLM_MODEL_NAME,

    "--host", "0.0.0.0",
    "--port", str(PORT),

    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),

    # Text-only mode skips the vision encoder and frees memory for KV cache.
    "--language-model-only",

    # Non-thinking mode for Qwen3/Qwen3.5.
    "--reasoning-parser", "qwen3",
    "--default-chat-template-kwargs", '{"enable_thinking": false}',

    "--max-num-seqs", str(MAX_NUM_SEQS),
    "--max-num-batched-tokens", str(MAX_NUM_BATCHED_TOKENS),

    "--enable-prefix-caching",
    "--generation-config", "vllm",
    "--dtype", "bfloat16",
    "--trust-remote-code",
]

server_env = os.environ.copy()

# Do not set Blackwell/CUDA-13 environment variables on A100.
# A100 should use the CUDA/PyTorch backend selected during installation.

print("Command:")
print(" ".join(shlex.quote(x) for x in cmd))

print("\nCUDA / GPU check:")
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

SERVER_LOG_PATH.write_text("", encoding="utf-8")
log_file = open(SERVER_LOG_PATH, "w", encoding="utf-8")

proc = subprocess.Popen(
    cmd,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
    env=server_env,
)

SERVER_PID_PATH.write_text(str(proc.pid))

print("\nStarted vLLM server.")
print("PID:", proc.pid)
print("Log:", SERVER_LOG_PATH)

Command:
vllm serve Qwen/Qwen3.5-27B --host 0.0.0.0 --port 8000 --max-model-len 10240 --gpu-memory-utilization 0.9 --language-model-only --reasoning-parser qwen3 --default-chat-template-kwargs '{"enable_thinking": false}' --max-num-seqs 1 --max-num-batched-tokens 10240 --enable-prefix-caching --generation-config vllm --dtype bfloat16 --trust-remote-code

CUDA / GPU check:
Torch CUDA: 13.0
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB

Started vLLM server.
PID: 6840
Log: /content/vllm_bm25_answer_server.log


In [6]:
#cell 6
# Wait for vLLM server and create an OpenAI-compatible client.

import requests

def tail_log(path: Path, n: int = 80) -> str:
    """Read the last n log lines."""
    if not path.exists():
        return ""
    lines = path.read_text(errors="ignore").splitlines()
    return "\n".join(lines[-n:])

ready = False
SERVER_MODEL_ID = None

MAX_WAIT_SEC = 1800
SLEEP_SEC = 5
PRINT_EVERY_SEC = 60

start = time.perf_counter()
last_print = -PRINT_EVERY_SEC

for step in range(MAX_WAIT_SEC // SLEEP_SEC):
    elapsed = int(time.perf_counter() - start)

    return_code = proc.poll()
    if return_code is not None:
        print(f"vLLM process exited. Return code: {return_code}")
        print("\n=== Last vLLM log lines ===")
        print(tail_log(SERVER_LOG_PATH, n=160))
        raise RuntimeError("vLLM server crashed or exited during startup.")

    try:
        h = requests.get(f"http://localhost:{PORT}/health", timeout=5)
        if h.status_code == 200:
            m = requests.get(f"{BASE_URL}/models", timeout=10)
            if m.status_code == 200:
                ready = True
                model_info = m.json()["data"][0]
                SERVER_MODEL_ID = model_info["id"]
                print("vLLM server is ready.")
                print("Model:", SERVER_MODEL_ID)
                print("Max model len:", model_info.get("max_model_len"))
                break
    except Exception:
        pass

    if elapsed - last_print >= PRINT_EVERY_SEC:
        last_print = elapsed
        print(f"Waiting... {elapsed}s")
        recent = tail_log(SERVER_LOG_PATH, n=12)
        if recent.strip():
            print(recent)
        print("-" * 80)

    time.sleep(SLEEP_SEC)

if not ready:
    print("\n=== Last vLLM log lines ===")
    print(tail_log(SERVER_LOG_PATH, n=160))
    raise RuntimeError("vLLM server did not become ready before timeout.")

client = OpenAI(
    api_key="EMPTY",
    base_url=BASE_URL,
    timeout=3600,
)

# Deterministic decoding is better for QA evaluation.
LLM_SAMPLING_KWARGS = {
    "temperature": 0.0,
    "top_p": 1.0,
    "presence_penalty": 0.0,
}

LLM_EXTRA_BODY = {
    "top_k": 20,
    "min_p": 0.0,
    "repetition_penalty": 1.0,
    "chat_template_kwargs": {
        "enable_thinking": False,
    },
}

print("OpenAI-compatible client is ready.")
print("Server model id:", SERVER_MODEL_ID)

Waiting... 0s
--------------------------------------------------------------------------------
Waiting... 60s
(APIServer pid=6840) INFO 06-15 14:53:43 [api_utils.py:339] 
(APIServer pid=6840) INFO 06-15 14:53:43 [api_utils.py:273] non-default args: {'model_tag': 'Qwen/Qwen3.5-27B', 'default_chat_template_kwargs': {'enable_thinking': False}, 'host': '0.0.0.0', 'model': 'Qwen/Qwen3.5-27B', 'trust_remote_code': True, 'dtype': 'bfloat16', 'max_model_len': 10240, 'generation_config': 'vllm', 'reasoning_parser': 'qwen3', 'gpu_memory_utilization': 0.9, 'enable_prefix_caching': True, 'language_model_only': True, 'max_num_batched_tokens': 10240, 'max_num_seqs': 1}
(APIServer pid=6840) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
(APIServer pid=6840) INFO 06-15 14:54:00 [model.py:598] Resolved architecture: Qwen3_5ForConditionalGeneration
(APIServer pid=6840) INFO 06-15 14:54:00 [model.py:1723] Using max

In [7]:
#cell 7
# Load tokenizer and define prompt/context builders.

tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL_NAME,
    trust_remote_code=True,
)

BASE_PROMPT_TEMPLATE = """
Given the question and its associated contexts below, please generate a concise, precise answer in English. The answer must strictly adhere to the following guidelines:

- The answer should be directly relevant to the question.
- Provide the answer in a clear, straightforward format.
- Limit your answer to no more than 6 words, focusing on the essential information requested.
- If the provided contexts do not contain enough information to answer the question, respond with "Information not available".
- Do not include any additional tokens, explanations, or information beyond the direct answer.

QUESTION: {question}
CONTEXT:
{context}

ANSWER:
""".strip()

def count_tokens(text: str) -> int:
    """Count tokens without adding special tokens."""
    return len(tokenizer.encode(text, add_special_tokens=False))

def format_context_from_chunks(evidence_chunks: List[Dict[str, Any]]) -> str:
    """
    Format only the retrieved evidence chunks.
    No answer, supports, or ground-truth fields are included.
    """
    parts = []

    for rank, chunk in enumerate(evidence_chunks, start=1):
        title = str(chunk.get("title", "")).strip()
        text = str(chunk.get("text", "")).strip()

        parts.append(
            f"[Chunk {rank}]\n"
            f"Title: {title}\n"
            f"Text: {text}"
        )

    return "\n\n".join(parts).strip()

def build_prompt_without_truncation(question: str, context: str) -> str:
    """Build the final user prompt."""
    return BASE_PROMPT_TEMPLATE.format(
        question=str(question).strip(),
        context=str(context).strip(),
    )

def build_prompt(question: str, evidence_chunks: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Build a prompt using only question and evidence_chunk.
    The context is truncated if the full prompt exceeds MAX_INPUT_TOKENS.
    """
    context = format_context_from_chunks(evidence_chunks)
    prompt = build_prompt_without_truncation(question, context)

    original_tokens = count_tokens(prompt)

    if original_tokens <= MAX_INPUT_TOKENS:
        return {
            "prompt": prompt,
            "input_tokens": original_tokens,
            "was_truncated": False,
        }

    # Compute available token budget for context after keeping the prompt shell.
    empty_prompt = build_prompt_without_truncation(question, "")
    overhead_tokens = count_tokens(empty_prompt)

    context_budget = max(0, MAX_INPUT_TOKENS - overhead_tokens - 16)

    context_ids = tokenizer.encode(context, add_special_tokens=False)
    truncated_context_ids = context_ids[:context_budget]
    truncated_context = tokenizer.decode(truncated_context_ids, skip_special_tokens=True)

    truncated_prompt = build_prompt_without_truncation(question, truncated_context)
    truncated_tokens = count_tokens(truncated_prompt)

    return {
        "prompt": truncated_prompt,
        "input_tokens": truncated_tokens,
        "was_truncated": True,
        "original_input_tokens": original_tokens,
    }

# Sanity check: make sure the prompt does not contain forbidden fields by construction.
sample_dataset = "hotpotqa"
sample_record = evidence_data[sample_dataset][0]
sample_prompt_info = build_prompt(
    question=sample_record["question"],
    evidence_chunks=sample_record["evidence_chunk"],
)

print("Sample dataset:", sample_dataset)
print("Sample input tokens:", sample_prompt_info["input_tokens"])
print("Sample was truncated:", sample_prompt_info["was_truncated"])
print("\nSample prompt preview:")
print(sample_prompt_info["prompt"][:2000])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Sample dataset: hotpotqa
Sample input tokens: 1875
Sample was truncated: False

Sample prompt preview:
Given the question and its associated contexts below, please generate a concise, precise answer in English. The answer must strictly adhere to the following guidelines:

- The answer should be directly relevant to the question.
- Provide the answer in a clear, straightforward format.
- Limit your answer to no more than 6 words, focusing on the essential information requested.
- If the provided contexts do not contain enough information to answer the question, respond with "Information not available".
- Do not include any additional tokens, explanations, or information beyond the direct answer.

QUESTION: Which musical fantasy film is older, Bedknobs and Broomsticks or The Muppet Christmas Carol?
CONTEXT:
[Chunk 1]
Title: The Muppet Christmas Carol
Text: The Muppet Christmas Carol is a 1992 American-British musical fantasy comedy-drama film and an adaptation of Charles Dickens's 1843 n

In [8]:
#cell 8
# Define LLM call, response cleanup, and JSON saving helpers.

def clean_model_response(text: str) -> str:
    """Clean possible reasoning or formatting artifacts from the model output."""
    if text is None:
        return ""

    text = str(text)

    # Remove any accidental thinking block if the model emits one.
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE)
    text = text.replace("<think>", "").replace("</think>", "")

    text = text.strip()

    # Remove common labels if the model adds them.
    text = re.sub(r"^\s*ANSWER\s*:\s*", "", text, flags=re.IGNORECASE).strip()

    # Keep only the first non-empty line.
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if lines:
        text = lines[0].strip()

    # Remove surrounding quotes.
    text = text.strip().strip('"').strip("'").strip()

    return text

def call_llm_answer(prompt: str, max_retries: int = 3) -> str:
    """Call the local vLLM OpenAI-compatible server and return a cleaned answer."""
    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            completion = client.chat.completions.create(
                model=SERVER_MODEL_ID or LLM_MODEL_NAME,
                messages=[
                    {
                        "role": "user",
                        "content": prompt,
                    }
                ],
                max_tokens=ANSWER_MAX_TOKENS,
                **LLM_SAMPLING_KWARGS,
                extra_body=LLM_EXTRA_BODY,
            )

            raw_text = completion.choices[0].message.content
            return clean_model_response(raw_text)

        except Exception as e:
            last_error = e
            print(f"LLM call failed on attempt {attempt}/{max_retries}: {repr(e)}")
            time.sleep(2 * attempt)

    raise RuntimeError(f"LLM call failed after {max_retries} attempts: {repr(last_error)}")

def atomic_write_json(data: Any, path: Path) -> None:
    """Atomically save JSON to the target path."""
    path.parent.mkdir(parents=True, exist_ok=True)

    tmp_path = path.with_name(path.name + ".tmp")

    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    os.replace(tmp_path, path)

def load_existing_answers(answer_path: Path) -> List[Dict[str, Any]]:
    """Load existing answers for resume mode."""
    if not answer_path.exists():
        return []

    with open(answer_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(f"Existing answer file must contain a list: {answer_path}")

    return data

def validate_answer_record(record: Dict[str, Any], dataset_name: str, index: int) -> None:
    """Validate one answer output record."""
    required_keys = {"type", "question", "gt", "response"}

    if set(record.keys()) != required_keys:
        raise ValueError(
            f"{dataset_name}: answer record {index} must have exactly {required_keys}, "
            f"but found {set(record.keys())}"
        )

print("Helper functions are ready.")

Helper functions are ready.


In [9]:
#cell 9
# Test one LLM call before running the full datasets.

test_dataset = "hotpotqa"
test_record = evidence_data[test_dataset][0]

test_prompt_info = build_prompt(
    question=test_record["question"],
    evidence_chunks=test_record["evidence_chunk"],
)

print("Test dataset:", test_dataset)
print("Test question:", test_record["question"])
print("Test GT answer:", test_record["answer"])
print("Test input tokens:", test_prompt_info["input_tokens"])
print("Test was truncated:", test_prompt_info["was_truncated"])

test_response = call_llm_answer(test_prompt_info["prompt"])

print("Test LLM response:", test_response)

Test dataset: hotpotqa
Test question: Which musical fantasy film is older, Bedknobs and Broomsticks or The Muppet Christmas Carol?
Test GT answer: Bedknobs and Broomsticks
Test input tokens: 1875
Test was truncated: False
Test LLM response: Bedknobs and Broomsticks


In [10]:
#cell 10
# Process one dataset and save answers.

def get_processing_slice(records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Return the selected processing slice."""
    end = ANSWER_END_INDEX if ANSWER_END_INDEX is not None else len(records)
    return records[ANSWER_START_INDEX:end]

def process_dataset(dataset_name: str, records: List[Dict[str, Any]], answer_path: Path) -> Dict[str, Any]:
    """
    Process one dataset independently.

    Important:
    - Only question and evidence_chunk are given to the LLM.
    - answer/supports/type are not included in the prompt.
    - The final saved JSON contains exactly: type, question, gt, response.
    """
    print("=" * 80)
    print(f"Starting dataset: {dataset_name}")
    print("Answer path:", answer_path)

    selected_records = get_processing_slice(records)

    if not selected_records:
        raise ValueError(f"{dataset_name}: selected record slice is empty.")

    answer_records = []

    if RESUME_IF_EXISTS and answer_path.exists():
        existing = load_existing_answers(answer_path)

        # Resume only if the existing file matches the selected prefix.
        can_resume = True

        if len(existing) > len(selected_records):
            can_resume = False
        else:
            for i, old in enumerate(existing):
                if old.get("question") != selected_records[i].get("question"):
                    can_resume = False
                    break

        if can_resume:
            answer_records = existing
            print(f"{dataset_name}: resuming from {len(answer_records)} existing answers.")
        else:
            print(f"{dataset_name}: existing answer file does not match current data. Starting over.")

    start_time = time.time()

    input_token_counts = []
    truncation_count = 0

    start_i = len(answer_records)

    for local_i in tqdm(
        range(start_i, len(selected_records)),
        desc=f"Answering {dataset_name}"
    ):
        rec = selected_records[local_i]

        prompt_info = build_prompt(
            question=rec["question"],
            evidence_chunks=rec["evidence_chunk"],
        )

        input_token_counts.append(prompt_info["input_tokens"])
        if prompt_info.get("was_truncated", False):
            truncation_count += 1

        response = call_llm_answer(prompt_info["prompt"])

        output_record = {
            "type": rec["type"],
            "question": rec["question"],
            "gt": rec["answer"],
            "response": response,
        }

        validate_answer_record(output_record, dataset_name, local_i)
        answer_records.append(output_record)

        if len(answer_records) % SAVE_EVERY_N == 0:
            atomic_write_json(answer_records, answer_path)

        if CLEAR_CACHE_EVERY_N and len(answer_records) % CLEAR_CACHE_EVERY_N == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    atomic_write_json(answer_records, answer_path)

    elapsed = time.time() - start_time

    # Validate saved file.
    saved = load_existing_answers(answer_path)

    if len(saved) != len(selected_records):
        raise ValueError(
            f"{dataset_name}: saved {len(saved)} answers, "
            f"but expected {len(selected_records)}."
        )

    for i, out_rec in enumerate(saved):
        validate_answer_record(out_rec, dataset_name, i)

    summary = {
        "dataset": dataset_name,
        "num_input_records": len(records),
        "num_processed_records": len(saved),
        "answer_path": str(answer_path),
        "elapsed_seconds": elapsed,
        "num_truncated_prompts": truncation_count,
        "max_input_tokens_seen": max(input_token_counts) if input_token_counts else None,
        "mean_input_tokens_seen": float(sum(input_token_counts) / len(input_token_counts)) if input_token_counts else None,
    }

    print("=" * 80)
    print(f"{dataset_name}: done.")
    print("Saved to:", answer_path)
    print("Processed records:", len(saved))
    print("Elapsed seconds:", elapsed)
    print("Truncated prompts:", truncation_count)
    print("First output record:", saved[0])

    return summary

print("Dataset processing function is ready.")

Dataset processing function is ready.


In [11]:
#cell 11
# Run HotpotQA BM25 dataset separately and save its independent JSON answer file.

run_summaries = []

hotpotqa_summary = process_dataset(
    dataset_name="hotpotqa",
    records=evidence_data["hotpotqa"],
    answer_path=DATASETS["hotpotqa"]["answer_path"],
)

# Keep only one summary per dataset if this cell is re-run.
run_summaries = [s for s in run_summaries if s.get("dataset") != "hotpotqa"]
run_summaries.append(hotpotqa_summary)

summary_df = pd.DataFrame(run_summaries)
display(summary_df)

print("HotpotQA BM25 answering is complete.")
print("Saved to:", DATASETS["hotpotqa"]["answer_path"])

Starting dataset: hotpotqa
Answer path: /content/drive/MyDrive/final_project/BM25/answer/hotpotqa_qwen3.5_bm25_answers.json


Answering hotpotqa:   0%|          | 0/1000 [00:00<?, ?it/s]

hotpotqa: done.
Saved to: /content/drive/MyDrive/final_project/BM25/answer/hotpotqa_qwen3.5_bm25_answers.json
Processed records: 1000
Elapsed seconds: 675.2806255817413
Truncated prompts: 0
First output record: {'type': 'comparison', 'question': 'Which musical fantasy film is older, Bedknobs and Broomsticks or The Muppet Christmas Carol?', 'gt': 'Bedknobs and Broomsticks', 'response': 'Bedknobs and Broomsticks'}


,dataset,num_input_records,num_processed_records,answer_path,elapsed_seconds,num_truncated_prompts,max_input_tokens_seen,mean_input_tokens_seen
0,hotpotqa,1000,1000,/content/drive/MyDrive/final_project/BM25/answ...,675.280626,0,2351,1683.524


HotpotQA BM25 answering is complete.
Saved to: /content/drive/MyDrive/final_project/BM25/answer/hotpotqa_qwen3.5_bm25_answers.json


In [12]:
#cell 12
# Run 2WikiMultihopQA BM25 dataset separately and save its independent JSON answer file.

try:
    run_summaries
except NameError:
    run_summaries = []

twiki_summary = process_dataset(
    dataset_name="2wikimultihopqa",
    records=evidence_data["2wikimultihopqa"],
    answer_path=DATASETS["2wikimultihopqa"]["answer_path"],
)

# Keep only one summary per dataset if this cell is re-run.
run_summaries = [s for s in run_summaries if s.get("dataset") != "2wikimultihopqa"]
run_summaries.append(twiki_summary)

summary_df = pd.DataFrame(run_summaries)
display(summary_df)

print("2WikiMultihopQA BM25 answering is complete.")
print("Saved to:", DATASETS["2wikimultihopqa"]["answer_path"])

Starting dataset: 2wikimultihopqa
Answer path: /content/drive/MyDrive/final_project/BM25/answer/2wikimultihopqa_qwen3.5_bm25_answers.json


Answering 2wikimultihopqa:   0%|          | 0/1000 [00:00<?, ?it/s]

2wikimultihopqa: done.
Saved to: /content/drive/MyDrive/final_project/BM25/answer/2wikimultihopqa_qwen3.5_bm25_answers.json
Processed records: 1000
Elapsed seconds: 660.7311544418335
Truncated prompts: 0
First output record: {'type': 'bridge_comparison', 'question': 'Which film has the director died earlier, John Jaffer Janardhanan or Kamakalawa?', 'gt': 'Kamakalawa', 'response': 'Information not available'}


,dataset,num_input_records,num_processed_records,answer_path,elapsed_seconds,num_truncated_prompts,max_input_tokens_seen,mean_input_tokens_seen
0,hotpotqa,1000,1000,/content/drive/MyDrive/final_project/BM25/answ...,675.280626,0,2351,1683.524
1,2wikimultihopqa,1000,1000,/content/drive/MyDrive/final_project/BM25/answ...,660.731154,0,2307,1538.374


2WikiMultihopQA BM25 answering is complete.
Saved to: /content/drive/MyDrive/final_project/BM25/answer/2wikimultihopqa_qwen3.5_bm25_answers.json


In [13]:
#cell 13
# Verify final BM25 output files.

def verify_final_answer_file(dataset_name: str, answer_path: Path) -> None:
    """Verify final answer JSON format."""
    records = load_existing_answers(answer_path)

    if not records:
        raise ValueError(f"{dataset_name}: answer file is empty: {answer_path}")

    for i, rec in enumerate(records):
        validate_answer_record(rec, dataset_name, i)

    print("=" * 80)
    print(f"{dataset_name}: final BM25 answer file verified.")
    print("Path:", answer_path)
    print("Number of records:", len(records))
    print("First record keys:", list(records[0].keys()))
    print("First question:", records[0]["question"])
    print("First GT:", records[0]["gt"])
    print("First response:", records[0]["response"])

for dataset_name, cfg in DATASETS.items():
    verify_final_answer_file(dataset_name, cfg["answer_path"])

print("\nBM25 answer folder contents:")
for item in sorted(DRIVE_ANSWER_DIR.iterdir()):
    print(" -", item.name)

hotpotqa: final BM25 answer file verified.
Path: /content/drive/MyDrive/final_project/BM25/answer/hotpotqa_qwen3.5_bm25_answers.json
Number of records: 1000
First record keys: ['type', 'question', 'gt', 'response']
First question: Which musical fantasy film is older, Bedknobs and Broomsticks or The Muppet Christmas Carol?
First GT: Bedknobs and Broomsticks
First response: Bedknobs and Broomsticks
2wikimultihopqa: final BM25 answer file verified.
Path: /content/drive/MyDrive/final_project/BM25/answer/2wikimultihopqa_qwen3.5_bm25_answers.json
Number of records: 1000
First record keys: ['type', 'question', 'gt', 'response']
First question: Which film has the director died earlier, John Jaffer Janardhanan or Kamakalawa?
First GT: Kamakalawa
First response: Information not available

BM25 answer folder contents:
 - 2wikimultihopqa_qwen3.5_bm25_answers.json
 - hotpotqa_qwen3.5_bm25_answers.json
